# OASIS simulation baseline

Run this notebook independently of `lpcmci.ipynb`. Prepare the shared project environment once with `uv sync --frozen --all-extras`; the committed notebook extras preserve the required kernel tools, and selecting all extras prevents one baseline sync from removing the other. OASIS is evaluated as an event estimator and as preprocessing for matched c-GC and c-GC* graph recovery. After the 12,000-fit matched grid, the notebook automatically runs the c-GC/c-GC* H1-H4 analysis suite: representative recovery, locked rise/fall tests, cyclic-shift and reverse-time nulls, the falling comparator, W_IC contrasts, and noise/frame-rate sweeps.

<!-- reviewer-resume-contract -->
## Execution and resume contract

Each outer-run–condition–seed unit is checkpointed. It recreates the exact static c-GC/c-GC* network and fluorescence input and records an input digest. Re-run the identical cell after interruption; do not change the grid, representations, or output directory while resuming. The main progress file ends at 1,000/1,000 units, 12,000 graph rows, and 4,000 event rows; `full_analysis/progress.json` then ends at 225/225 diagnostic units and 450/450 downstream fits. If an old five-representation OASIS output is found, it is preserved beside the output directory and the corrected six-representation run starts cleanly.

In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys


def find_package_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for base in (current, *current.parents):
        for candidate in (base, base / 'calcium-transient-rising-flank'):
            if (candidate / 'pyproject.toml').is_file() and (candidate / 'examples' / 'simulation_baselines.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate the calcium-transient-rising-flank package root')


PACKAGE_ROOT = find_package_root()
RUNNER_PYTHON = next((str(path) for path in (PACKAGE_ROOT / '.venv/bin/python', PACKAGE_ROOT / '.venv/Scripts/python.exe') if path.is_file()), sys.executable)
RUNNER_ENV = os.environ.copy()
RUNNER_ENV['MPLBACKEND'] = 'Agg'
RUNNER_ENV['PYTHONUNBUFFERED'] = '1'
SOURCE_ROOT = str(PACKAGE_ROOT / 'src')
RUNNER_ENV['PYTHONPATH'] = os.pathsep.join(filter(None, (SOURCE_ROOT, RUNNER_ENV.get('PYTHONPATH'))))
sys.path.insert(0, SOURCE_ROOT)

from calcium_transient_rising_flank.checkpointing import format_progress

NOTEBOOK_TAG = 'simulations/oasis.ipynb'
RUN_OASIS = True
N_RUNS_OUTER = 10
N_SEEDS = 20
N_STEPS = 3000
N_SURROGATES = 1000
N_NULL = 6
REPRESENTATIONS = 'full,deconvolved,oasis,rise,fall,fall_residual'
OUTPUT_DIR = PACKAGE_ROOT / 'outputs/revision_campaign/oasis_simulation'
FULL_ANALYSIS_DIR = OUTPUT_DIR / 'full_analysis'
command = [
    RUNNER_PYTHON, 'examples/simulation_baselines.py',
    '--components', 'oasis',
    '--representations', REPRESENTATIONS,
    '--cgc-methods', 'cgc,cgc-star',
    '--n-runs-outer', str(N_RUNS_OUTER), '--n-seeds', str(N_SEEDS),
    '--n-steps', str(N_STEPS), '--n-cgc-surrogates', str(N_SURROGATES),
    '--output-dir', str(OUTPUT_DIR), '--resume', '--restart-incompatible-resume',
]
analysis_command = [
    RUNNER_PYTHON, 'examples/simulation_baseline_diagnostics.py',
    '--baseline', 'oasis', '--baseline-dir', str(OUTPUT_DIR),
    '--output-dir', str(FULL_ANALYSIS_DIR), '--n-null', str(N_NULL), '--resume',
]
progress_path = OUTPUT_DIR / 'progress.json'
analysis_progress_path = FULL_ANALYSIS_DIR / 'progress.json'
total_units = N_RUNS_OUTER * 5 * N_SEEDS
total_fits = total_units * len(REPRESENTATIONS.split(',')) * 2
analysis_units = len(REPRESENTATIONS.split(',')) + (N_NULL + 3) + N_RUNS_OUTER * min(3, N_SEEDS) * 7
analysis_fits = analysis_units * 2


def notebook_log(status: str, message: str) -> None:
    print(f'[{NOTEBOOK_TAG}] {status}: {message}', flush=True)


def show_resume_state(*, label: str, fallback_total: int, path: Path) -> None:
    if path.exists():
        saved = json.loads(path.read_text())
        completed = int(saved.get('completed_unit_count') or 0)
        expected = int(saved.get('expected_unit_count') or fallback_total or 1)
        total = max(expected, 1)
        notebook_log('RESUMED', format_progress(min(completed, total), total, label=label))
        notebook_log(
            'RESUMED',
            f"status={saved.get('status')} active={saved.get('active_unit')} progress_file={path}",
        )
    else:
        notebook_log('START', format_progress(0, max(fallback_total, 1), label=label))
        notebook_log('START', f'no saved progress at {path}')


notebook_log('START', f'output={OUTPUT_DIR}')
notebook_log('START', f'planned work={total_units} checkpoint units / {total_fits} downstream graph fits')
show_resume_state(label='OASIS checkpoint units', fallback_total=total_units, path=progress_path)
notebook_log('START', f'launching: {shlex.join(command)}')
if RUN_OASIS:
    subprocess.run(command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)
    show_resume_state(label='OASIS checkpoint units', fallback_total=total_units, path=progress_path)
    notebook_log('DONE', 'matched-grid runner finished successfully')
    notebook_log('START', f'full H1-H4 analysis planned work={analysis_units} units / {analysis_fits} downstream fits')
    show_resume_state(label='OASIS full-analysis units', fallback_total=analysis_units, path=analysis_progress_path)
    notebook_log('START', f'launching: {shlex.join(analysis_command)}')
    subprocess.run(analysis_command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)
    show_resume_state(label='OASIS full-analysis units', fallback_total=analysis_units, path=analysis_progress_path)
    notebook_log('DONE', 'full H1-H4 analysis finished successfully')

summary_path = OUTPUT_DIR / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))
analysis_summary_path = FULL_ANALYSIS_DIR / 'summary.json'
if analysis_summary_path.exists():
    analysis_summary = json.loads(analysis_summary_path.read_text())
    print(json.dumps(analysis_summary, indent=2))


## Analysis sections

The cells below mirror the named analyses in `c-GC.ipynb` and `c-GC-star.ipynb`, using OASIS-inferred spikes as the OASIS representation. OASIS itself is a deconvolution method, not a graph learner; therefore event-recovery results are labeled `oasis`, while graph recovery and $W_{IC}$ remain explicitly separated by downstream `cgc` and `cgc-star` in the `method` column.

In [ ]:
import pandas as pd
from IPython.display import Image, display


def show_analysis_table(filename: str) -> pd.DataFrame:
    path = FULL_ANALYSIS_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f'Missing analysis table: {path}')
    frame = pd.read_csv(path)
    display(frame.round(4))
    return frame


def show_analysis_figure(filename: str) -> None:
    path = FULL_ANALYSIS_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f'Missing analysis figure: {path}')
    display(Image(filename=str(path)))


## 1. H1 — kinetic asymmetry and transient characterization

Per-ROI decay persistence, event density, rise count, median rise duration, and robust SNR on the fixed bilateral-chain simulation.

In [ ]:
h1 = show_analysis_table('h1_transient_characterization.csv')
show_analysis_figure('waveform.png')
show_analysis_figure('h1_transient.png')


## 2. H2 — representation recovery and locked condition analysis

Representative recovery is compared across full, deconvolved, OASIS, rise, fall, and fall-residual inputs. The locked table uses only evaluation seeds from the five-condition matched grid, followed by paired rise-minus-fall sign-flip tests kept separate for downstream c-GC and c-GC*.

In [ ]:
representative_recovery = show_analysis_table('representative_recovery.csv')
locked_recovery = show_analysis_table('locked_recovery_by_representation.csv')
locked_by_condition = show_analysis_table('locked_recall_by_condition.csv')
paired_rise_fall = show_analysis_table('paired_rise_vs_fall_test.csv')
oasis_event_recovery = show_analysis_table('oasis_event_diagnostics.csv')
show_analysis_figure('h2_representations.png')
show_analysis_figure('h4_conditions.png')


## 3. Null and negative-comparator checks

The observed OASIS spike representation is compared with six independent cyclic shifts, reversed time, and the falling-flank comparator. Directed graph results remain separated by the downstream `method` value; OASIS event recovery remains separately labeled.

In [ ]:
null_and_negative_comparators = show_analysis_table('null_comparator_rows.csv')
show_analysis_figure('h3_nulls.png')


## 4. Ipsilateral consistency and $\Delta W_{\mathrm{IC}}$

Paired rise-minus-fall ipsilateral-consistency contrasts are calculated for every matched run, condition, and seed and kept separate for downstream c-GC and c-GC*.

In [ ]:
wic_delta_rows = show_analysis_table('wic_delta_rows.csv')
wic_delta_by_condition = show_analysis_table('wic_delta_by_condition.csv')
show_analysis_figure('wic_delta.png')


## 5. H4 — robustness to noise and frame-rate reduction

The OASIS spike representation is evaluated at four observation-noise levels and three downsampling factors over 10 networks and three seeds per network, with downstream graph results kept separate by method.

In [ ]:
robustness_noise = show_analysis_table('robustness_noise.csv')
robustness_framerate = show_analysis_table('robustness_framerate.csv')
show_analysis_figure('h4_sweeps.png')
